# Transfer Learning — Clasificación de Neumonía

Este notebook implementa **Transfer Learning** usando ResNet18 preentrenado en ImageNet.

**Estrategia de dos fases:**
1. **Feature Extraction** (5 épocas): Capas base congeladas
2. **Fine-tuning** (3 épocas): Descongelación de layer4

**Dataset:** Chest X-Ray Images (Pneumonia) desde Kaggle

## 1️⃣ Configuración del Entorno

Como en el notebook anterior, necesitamos preparar todas nuestras herramientas antes de empezar. La diferencia aquí es que usaremos modelos preentrenados de TorchVision, así que necesitamos importar esa funcionalidad.

En esta sección vamos a:
- **Instalar las librerías necesarias** (PyTorch, TorchVision, etc.)
- **Importar todas las herramientas** incluyendo modelos preentrenados
- **Configurar el dispositivo** (GPU o CPU) para un entrenamiento rápido
- **Establecer semillas aleatorias** para reproducibilidad

### Código

In [ ]:
import importlib.util
import random
import subprocess
import sys
from pathlib import Path

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        package_name = pip_name or import_name
        print(f'Instalando: {package_name}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

ensure_package('kagglehub')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from PIL import Image
import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 2️⃣ Transformaciones de Imágenes

ResNet18 fue entrenado originalmente con imágenes de ImageNet, un dataset de objetos cotidianos. Para que funcione bien con nuestras radiografías, necesitamos preparar las imágenes exactamente como lo hizo el modelo original.

En esta sección vamos a:
- **Redimensionar a 224×224 píxeles** (el tamaño que ResNet18 espera)
- **Convertir a tensores** que PyTorch pueda procesar
- **Normalizar con estadísticas de ImageNet** (los mismos valores que se usaron para entrenar ResNet18 originalmente)
- **Aplicar data augmentation** (solo en entrenamiento) para mejorar la robustez del modelo

Esto es crucial porque si no usamos las estadísticas correctas, el modelo no entenderá bien nuestras imágenes.

### Código

In [ ]:
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

print('Transformaciones definidas')

## 3️⃣ Descarga y Detección del Dataset

Al igual que en el notebook anterior, necesitamos obtener nuestro dataset de radiografías. Este proceso es idéntico: descargamos desde Kaggle y buscamos inteligentemente las carpetas correctas.

En esta sección vamos a:
- **Descargar el dataset** desde Kaggle (si no lo has descargado antes, reutilizaremos la descarga anterior)
- **Buscar automáticamente** las carpetas train, val y test
- **Verificar que todo esté listo** antes de continuar

### Código

In [ ]:
import os
import zipfile
import kagglehub

DATASET_HANDLE = 'paultimothymooney/chest-xray-pneumonia'
PROJECT_DIR = Path.cwd()
DOWNLOAD_DIR = PROJECT_DIR / 'data' / 'raw'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

try:
    downloaded_location = kagglehub.dataset_download(DATASET_HANDLE, output_dir=str(DOWNLOAD_DIR))
except TypeError:
    downloaded_location = kagglehub.dataset_download(DATASET_HANDLE)

downloaded_path = Path(downloaded_location).expanduser().resolve()

def find_dataset_root(search_root):
    required = {'train', 'val', 'test'}
    search_root = Path(search_root).resolve()
    
    if not search_root.exists() or search_root.is_file():
        if search_root.is_file():
            search_root = search_root.parent
        else:
            return None, []
    
    for current, dirnames, _ in os.walk(search_root):
        current_path = Path(current)
        children = {dirname.lower(): current_path / dirname for dirname in dirnames}
        
        if required.issubset(children):
            return (current_path, children['train'], children['val'], children['test']), []
    
    return None, []

search_roots = [downloaded_path if downloaded_path.is_dir() else downloaded_path.parent, DOWNLOAD_DIR.resolve()]
dataset_structure = None

for search_root in search_roots:
    result, _ = find_dataset_root(search_root)
    if result is not None:
        dataset_structure = result
        break

if dataset_structure is None:
    raise FileNotFoundError('Dataset no encontrado')

DATA_DIR, train_dir, val_dir, test_dir = dataset_structure
print(f'Dataset root: {DATA_DIR}')

## 4️⃣ Carga de Datasets y Repartición

Ahora cargamos los datos con las transformaciones que definimos. Como el conjunto de validación original es muy pequeño (solo 16 imágenes), vamos a repartir el conjunto de entrenamiento en 80% para entrenar y 20% para validar, igual que en el notebook anterior.

En esta sección vamos a:
- **Cargar los datasets** con ImageFolder de TorchVision
- **Aplicar las transformaciones** que definimos (redimensionamiento, normalización, etc.)
- **Repartir el conjunto de entrenamiento** en 80/20 para tener suficientes datos de validación
- **Crear DataLoaders** que empaquetan las imágenes en lotes para un entrenamiento eficiente

### Código

In [ ]:
train_dataset = datasets.ImageFolder(root=str(train_dir), transform=train_transform)
val_dataset = datasets.ImageFolder(root=str(val_dir), transform=eval_transform)
test_dataset = datasets.ImageFolder(root=str(test_dir), transform=eval_transform)

class_names = train_dataset.classes
num_classes = len(class_names)

train_size = int(0.8 * len(train_dataset))
indices = torch.randperm(len(train_dataset)).tolist()
train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f'Clases: {class_names}')
print(f'Train: {len(train_subset)} | Val: {len(val_subset)} | Test: {len(test_dataset)}')

## 5️⃣ Carga de ResNet18 Preentrenado

 En lugar de construir un modelo desde cero, vamos a usar ResNet18, un modelo que fue entrenado con millones de imágenes comunes. Este modelo ya sabe cómo detectar características visuales útiles (bordes, texturas, formas, etc.).

En esta sección vamos a:
- **Cargar ResNet18 con pesos preentrenados** de ImageNet
- **Congelar todas las capas** para que no se modifiquen durante el entrenamiento inicial
- **Reemplazar la capa final** para que prediga 2 clases (NORMAL y PNEUMONIA) en lugar de las 1000 clases de ImageNet
- **Enviar el modelo al dispositivo** (GPU o CPU)

Esto es mucho más eficiente que entrenar desde cero porque reutilizamos el conocimiento que ResNet18 ya tiene.

### Código

In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, num_classes)
model = model.to(device)

print('ResNet18 cargado y adaptado')

## 6️⃣ Función de Pérdida y Optimizador

Para entrenar nuestro modelo, necesitamos definir cómo va a aprender. Esto incluye una función que mida qué tan equivocado está (pérdida) y un algoritmo que corrija esos errores (optimizador).

En esta sección vamos a:
- **Definir CrossEntropyLoss** como nuestra función de pérdida (ideal para clasificación)
- **Crear un optimizador Adam** que solo entrena la capa final que reemplazamos
- **Usar una tasa de aprendizaje moderada** (0.001) para que el modelo aprenda sin cambios bruscos

Nota importante: Solo entrenamos la capa `fc` porque las otras capas están congeladas en esta fase.

### Código

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
print('Criterio y optimizador configurados')

## 7️⃣ Funciones de Entrenamiento y Evaluación

Vamos a definir dos funciones que usaremos en ambas fases del entrenamiento. Estas funciones encapsulan la lógica de entrenar y evaluar el modelo, haciéndolo más limpio y reutilizable.

En esta sección vamos a:
- **Definir train_one_epoch()** que entrena el modelo durante una época completa
- **Definir evaluate()** que evalúa el modelo sin actualizar pesos
- **Calcular métricas** como pérdida y precisión para monitorear el progreso

Estas funciones son idénticas a las del notebook anterior, pero las repetimos aquí para que el notebook sea independiente.

### Código

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return running_loss / total, correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    return running_loss / total, correct / total

print('Funciones definidas')

## 8️⃣ Fase 1: Feature Extraction (5 épocas)

En esta primera fase, vamos a entrenar solo la capa final que reemplazamos. Las capas convolucionales de ResNet18 permanecen congeladas, actuando como un extractor de características automático.

ResNet18 ya sabe cómo encontrar características visuales útiles en cualquier imagen. En esta fase, solo le enseñamos cómo combinar esas características para detectar neumonía.

En esta sección vamos a:
- **Entrenar durante 5 épocas** con las capas base congeladas
- **Monitorear el progreso** viendo cómo mejora la precisión
- **Usar la tasa de aprendizaje moderada** (0.001) que definimos

Este proceso es rápido porque solo estamos entrenando una pequeña capa al final.

### Código

In [ ]:
print('FASE 1: FEATURE EXTRACTION')
for epoch in range(5):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f'Epoch {epoch+1}/5 | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

print('Fase 1 completada')

## 9️⃣ Fase 2: Fine-tuning (3 épocas)

Después de que el modelo aprendió a combinar las características de ResNet18, vamos a descongelar la última capa convolucional (`layer4`) para permitir ajustes finos.

En esta sección vamos a:
- **Descongelar layer4** (la última capa convolucional de ResNet18)
- **Crear un nuevo optimizador** que solo entrena los parámetros descongelados
- **Usar una tasa de aprendizaje muy baja** (1e-5) para no destruir el conocimiento preentrenado
- **Entrenar durante 3 épocas** con este ajuste fino

Este proceso es más lento que la fase 1 porque estamos entrenando más capas, pero es crucial para adaptar el modelo al dominio médico.

### Código

In [ ]:
print('FASE 2: FINE-TUNING')
for name, param in model.named_parameters():
    if 'layer4' in name or 'fc' in name:
        param.requires_grad = True

optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)

for epoch in range(3):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    print(f'Epoch {epoch+1}/3 | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}')

print('Fase 2 completada')

## 🔟 Evaluación en Conjunto de Prueba

Después de entrenar el modelo en dos fases, llega el momento de la verdad. Vamos a evaluar nuestro modelo con imágenes que nunca ha visto antes para obtener una medida real de su desempeño.

En esta sección vamos a:
- **Usar el conjunto de prueba** que apartamos desde el principio
- **Evaluar el desempeño final** del modelo después de ambas fases de entrenamiento
- **Obtener métricas reales** de qué tan bien detecta neumonía

Si la precisión aquí es alta (idealmente 95%+), significa que nuestro Transfer Learning funcionó muy bien.

### Código

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'Test Accuracy: {test_acc:.4f}')

## 1️⃣1️⃣ Clasificación desde `images_sample` 

Ahora vamos a usar nuestro modelo entrenado con Transfer Learning para clasificar imágenes nuevas que tú proporciones. Este modelo ha sido mejorado con dos fases de entrenamiento, así que debería dar predicciones muy precisas.

En esta sección vamos a:
- **Buscar todas las imágenes** en la carpeta `images_sample/` que creaste en la raíz del proyecto
- **Procesar cada imagen** con las transformaciones correctas (redimensionamiento a 224×224 y normalización con estadísticas de ImageNet)
- **Hacer predicciones** usando nuestro modelo entrenado
- **Mostrar los resultados** con la clase predicha y la confianza

Lo mejor es que este código es **totalmente reutilizable**: cada vez que agregues nuevas imágenes a `images_sample/`, solo ejecuta esta celda de nuevo. No es necesario reentrenar

### Código

In [ ]:
def predict_single_image(image_path, model, class_names, device, transform):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f'No existe: {image_path}')
    
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img)
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, dim=1)
    
    predicted_class = class_names[predicted_idx.item()]
    confidence_value = confidence.item()
    return predicted_class, confidence_value

SAMPLE_IMAGES_DIR = Path.cwd() / 'images_sample'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png'}
SAMPLE_IMAGES_DIR.mkdir(exist_ok=True)

sample_images = []
if SAMPLE_IMAGES_DIR.exists():
    for img_file in sorted(SAMPLE_IMAGES_DIR.glob('*')):
        if img_file.suffix.lower() in IMAGE_EXTENSIONS:
            sample_images.append(img_file)

if not sample_images:
    print(f'Carpeta creada: {SAMPLE_IMAGES_DIR}')
    print('Coloca imágenes en esta carpeta y ejecuta de nuevo')
else:
    print(f'Clasificando {len(sample_images)} imágenes:')
    for img_path in sample_images:
        try:
            pred_class, confidence = predict_single_image(img_path, model, class_names, device, eval_transform)
            print(f'{img_path.name}: {pred_class} ({confidence*100:.2f}%)')
        except Exception as e:
            print(f'{img_path.name}: Error - {str(e)}')